In [3]:
import numpy as np
import pandas as pd

def run_module_5_risk_assessment(df_plan, risk_params=None):
    """
    MODULE 5: ĐÁNH GIÁ RỦI RO ĐA MỤC TIÊU VÀ NGẪU NHIÊN (KẾT HỢP BÀI 7 & BÀI 10)
    Đánh giá các rủi ro hệ thống (An ninh mạng, Môi trường, Phụ thuộc công nghệ/Việc làm)
    dưới góc nhìn xác suất (Stochastic Scenarios).
    """
    
    # 1. THIẾT LẬP THAM SỐ RỦI RO MẶC ĐỊNH (Tham chiếu Bài 7)
    if risk_params is None:
        risk_params = {
            'cyber_D': 0.45,   # Rủi ro mạng từ tăng trưởng Hạ tầng số
            'cyber_AI': 0.60,  # Rủi ro mạng từ AI
            'cyber_H': -0.35,  # Mức giảm rủi ro mạng nhờ có Nhân lực số
            
            'env_I': 0.50,     # Phát thải từ xây dựng hạ tầng phần cứng
            'env_AI': 0.40,    # Phát thải từ chạy server AI
            'env_D': -0.25,    # Giảm phát thải nhờ tối ưu hóa bằng công nghệ số
            
            'dep_AI': 0.70,    # Rủi ro phụ thuộc công nghệ lõi / mất việc do AI
            'dep_H': -0.40     # Khả năng tự chủ công nghệ nhờ đào tạo Nhân lực
        }
    
    # 2. XÂY DỰNG KHÔNG GIAN XÁC SUẤT (STOCHASTIC SCENARIOS - Tham chiếu Bài 10)
    scenarios = {
        "Lạc quan": {"probability": 0.20, "multiplier": 0.8}, 
        "Cơ sở":    {"probability": 0.60, "multiplier": 1.0}, 
        "Bi quan":  {"probability": 0.20, "multiplier": 1.5}  
    }
    
    # 3. TÍNH TOÁN RỦI RO QUA TỪNG NĂM THEO KẾ HOẠCH PHÂN BỔ
    years = sorted(df_plan["Năm"].unique())
    risk_records = []
    
    annual_budgets = df_plan.groupby("Năm")["Ngân sách (Tỷ VND)"].sum()
    
    for year in years:
        df_year = df_plan[df_plan["Năm"] == year]
        total_budget = annual_budgets[year]
        
        # Tính phần trăm phân bổ (0 -> 1)
        p_I = df_year[df_year["Hạng mục"].str.contains("Hạ tầng")]["Ngân sách (Tỷ VND)"].sum() / total_budget
        p_D = df_year[df_year["Hạng mục"].str.contains("Chuyển đổi")]["Ngân sách (Tỷ VND)"].sum() / total_budget
        p_AI = df_year[df_year["Hạng mục"].str.contains("Trí tuệ")]["Ngân sách (Tỷ VND)"].sum() / total_budget
        p_H = df_year[df_year["Hạng mục"].str.contains("Nhân lực")]["Ngân sách (Tỷ VND)"].sum() / total_budget
        
        # Hàm tính rủi ro nội sinh gốc (Base Risk)
        base_cyber = max(0, risk_params['cyber_D']*p_D + risk_params['cyber_AI']*p_AI + risk_params['cyber_H']*p_H)
        base_env   = max(0, risk_params['env_I']*p_I + risk_params['env_AI']*p_AI + risk_params['env_D']*p_D)
        base_dep   = max(0, risk_params['dep_AI']*p_AI + risk_params['dep_H']*p_H)
        
        # 4. ÁP DỤNG XÁC SUẤT VÀ TÍNH KỲ VỌNG (EXPECTED VALUE - EV)
        expected_cyber = 0
        expected_env = 0
        expected_dep = 0
        
        for sc_name, sc_data in scenarios.items():
            prob = sc_data["probability"]
            mult = sc_data["multiplier"]
            
            sc_cyber = base_cyber * mult
            sc_env   = base_env * mult
            sc_dep   = base_dep * mult
            
            expected_cyber += sc_cyber * prob
            expected_env   += sc_env * prob
            expected_dep   += sc_dep * prob
            
            risk_records.append({
                "Năm": year,
                "Kịch bản Rủi ro": sc_name,
                "Xác suất": prob,
                "Cyber Risk (An ninh mạng)": sc_cyber * 100, 
                "Env Risk (Môi trường/Phát thải)": sc_env * 100,
                "Dep Risk (Phụ thuộc công nghệ)": sc_dep * 100
            })
            
        risk_records.append({
            "Năm": year,
            "Kịch bản Rủi ro": "KỲ VỌNG (Expected)",
            "Xác suất": 1.0,
            "Cyber Risk (An ninh mạng)": expected_cyber * 100,
            "Env Risk (Môi trường/Phát thải)": expected_env * 100,
            "Dep Risk (Phụ thuộc công nghệ)": expected_dep * 100
        })

    df_risks = pd.DataFrame(risk_records)
    return df_risks

# =====================================================================
# CHẠY TÍCH HỢP ĐÚNG KẾT QUẢ ĐẦU RA THỰC TẾ MODULE 3 CỦA BẠN
# =====================================================================
if __name__ == "__main__":
    print("Đang nạp kết quả tối ưu thực tế từ Module 3 để phân tích rủi ro...")
    
    # Khai báo chính xác bảng dữ liệu đầu ra dựa trên kết quả tối ưu của bạn
    actual_m3_records = [
        # --- Năm 2026 (Tổng ngân sách: 50,000 tỷ) ---
        {"Năm": 2026, "Vùng": "Vùng_2", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 15000.00},
        {"Năm": 2026, "Vùng": "Vùng_3", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 2500.00},
        {"Năm": 2026, "Vùng": "Vùng_4", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 5000.00},
        {"Năm": 2026, "Vùng": "Vùng_6", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 15000.00},
        {"Năm": 2026, "Vùng": "Vùng_1", "Hạng mục": "Trí tuệ nhân tạo (AI)", "Ngân sách (Tỷ VND)": 5000.00},
        {"Năm": 2026, "Vùng": "Vùng_3", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 2500.00},
        {"Năm": 2026, "Vùng": "Vùng_5", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 5000.00},
        
        # --- Năm 2027 (Tổng ngân sách: 55,000 tỷ) ---
        {"Năm": 2027, "Vùng": "Vùng_2", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 16500.00},
        {"Năm": 2027, "Vùng": "Vùng_3", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 2750.00},
        {"Năm": 2027, "Vùng": "Vùng_4", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 5500.00},
        {"Năm": 2027, "Vùng": "Vùng_6", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 16500.00},
        {"Năm": 2027, "Vùng": "Vùng_1", "Hạng mục": "Trí tuệ nhân tạo (AI)", "Ngân sách (Tỷ VND)": 5500.00},
        {"Năm": 2027, "Vùng": "Vùng_3", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 2750.00},
        {"Năm": 2027, "Vùng": "Vùng_5", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 5500.00},

        # --- Năm 2028 (Tổng ngân sách: 60,000 tỷ) ---
        {"Năm": 2028, "Vùng": "Vùng_2", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 18000.00},
        {"Năm": 2028, "Vùng": "Vùng_3", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 3000.00},
        {"Năm": 2028, "Vùng": "Vùng_4", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 6000.00},
        {"Năm": 2028, "Vùng": "Vùng_6", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 18000.00},
        {"Năm": 2028, "Vùng": "Vùng_1", "Hạng mục": "Trí tuệ nhân tạo (AI)", "Ngân sách (Tỷ VND)": 6000.00},
        {"Năm": 2028, "Vùng": "Vùng_3", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 3000.00},
        {"Năm": 2028, "Vùng": "Vùng_5", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 6000.00},

        # --- Năm 2029 (Tổng ngân sách: 65,000 tỷ) ---
        {"Năm": 2029, "Vùng": "Vùng_2", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 19500.00},
        {"Năm": 2029, "Vùng": "Vùng_3", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 3250.00},
        {"Năm": 2029, "Vùng": "Vùng_4", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 6500.00},
        {"Năm": 2029, "Vùng": "Vùng_6", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 19500.00},
        {"Năm": 2029, "Vùng": "Vùng_1", "Hạng mục": "Trí tuệ nhân tạo (AI)", "Ngân sách (Tỷ VND)": 6500.00},
        {"Năm": 2029, "Vùng": "Vùng_3", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 3250.00},
        {"Năm": 2029, "Vùng": "Vùng_5", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 6500.00},

        # --- Năm 2030 (Tổng ngân sách: 70,000 tỷ) ---
        {"Năm": 2030, "Vùng": "Vùng_2", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 21000.00},
        {"Năm": 2030, "Vùng": "Vùng_3", "Hạng mục": "Hạ tầng số (I)", "Ngân sách (Tỷ VND)": 3500.00},
        {"Năm": 2030, "Vùng": "Vùng_4", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 7000.00},
        {"Năm": 2030, "Vùng": "Vùng_6", "Hạng mục": "Chuyển đổi số DN (D)", "Ngân sách (Tỷ VND)": 21000.00},
        {"Năm": 2030, "Vùng": "Vùng_1", "Hạng mục": "Trí tuệ nhân tạo (AI)", "Ngân sách (Tỷ VND)": 7000.00},
        {"Năm": 2030, "Vùng": "Vùng_3", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 3500.00},
        {"Năm": 2030, "Vùng": "Vùng_5", "Hạng mục": "Nhân lực số (H)", "Ngân sách (Tỷ VND)": 7000.00},
    ]
    df_m3_plan = pd.DataFrame(actual_m3_records)
    
    # THỰC THI MODULE 5 TRÊN DỮ LIỆU THỰC TẾ
    df_m5_risks = run_module_5_risk_assessment(df_m3_plan)
    
    # TRÍCH XUẤT VÀ IN BÁO CÁO CẢNH BÁO
    df_expected = df_m5_risks[df_m5_risks["Kịch bản Rủi ro"] == "KỲ VỌNG (Expected)"]
    
    print("\n" + "="*85)
    print("BÁO CÁO CẢNH BÁO RỦI RO HỆ THỐNG KỲ VỌNG (THANG ĐIỂM 100) - TỪ 2026 ĐẾN 2030:")
    print("="*85)
    pd.set_option('display.float_format', lambda x: '%.2f' % x)
    print(df_expected[["Năm", "Cyber Risk (An ninh mạng)", "Env Risk (Môi trường/Phát thải)", "Dep Risk (Phụ thuộc công nghệ)"]].to_string(index=False))
    print("="*85)
    
    # Phân tích nhanh để đưa ra cảnh báo tự động
    avg_cyber = df_expected["Cyber Risk (An ninh mạng)"].mean()
    if avg_cyber > 25:
        print(f"\n[!] CẢNH BÁO: Rủi ro An ninh mạng khá cao ({avg_cyber:.2f}). Khuyến nghị Module 3 xem xét tăng phân bổ cho Nhân lực số (H) để tạo phòng tuyến an toàn.")
    else:
        print("\n[OK] CHÚC MỪNG: Các chỉ số rủi ro đều nằm trong ngưỡng an toàn nhờ chiến lược phân bổ cân bằng.")

Đang nạp kết quả tối ưu thực tế từ Module 3 để phân tích rủi ro...

BÁO CÁO CẢNH BÁO RỦI RO HỆ THỐNG KỲ VỌNG (THANG ĐIỂM 100) - TỪ 2026 ĐẾN 2030:
 Năm  Cyber Risk (An ninh mạng)  Env Risk (Môi trường/Phát thải)  Dep Risk (Phụ thuộc công nghệ)
2026                      19.88                            12.19                            1.06
2027                      19.88                            12.19                            1.06
2028                      19.88                            12.19                            1.06
2029                      19.88                            12.19                            1.06
2030                      19.88                            12.19                            1.06

[OK] CHÚC MỪNG: Các chỉ số rủi ro đều nằm trong ngưỡng an toàn nhờ chiến lược phân bổ cân bằng.
